# Greedy Choice and Correctness

```{contents}
:local:
:depth: 2
```

A greedy algorithm chooses locally and commits. Specify the objective, feasible choices, tie policy, and reason a choice can belong to an optimum. Feasibility alone is insufficient.

## Schedule the Maximum Number of Meetings

```{index} Schedule the Maximum Number of Meetings
```

One room holds fixed, positive-duration meetings [start,end). Back-to-back meetings are compatible. Maximize count; every meeting has equal importance for this objective. Select compatible meetings in nondecreasing finish order. Validate unique IDs and Start < End; empty input returns an empty selection.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
var chosen=MeetingSchedule.Select(meetings);
Console.WriteLine(string.Join(",",chosen.Select(m=>m.Id)));
Console.WriteLine($"count={chosen.Length}");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


The selected meetings never overlap, and lastEnd is the end of the last selected meeting. Sorting costs Θ(n log n); the selection scan is Θ(n). This implementation returns up to n selected records and uses sorting storage, so it does not claim constant total auxiliary space. Equal finish times use deterministic start/ID tie breaks; another compatible tie rule can also achieve the optimal count.

## Trace Accepted and Rejected Choices

```{index} Trace Accepted and Rejected Choices
```

A useful trace includes rejected choices and the evolving feasibility boundary.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",1,4),new MeetingSchedule.Meeting("B",3,5),new MeetingSchedule.Meeting("C",0,6),new MeetingSchedule.Meeting("D",5,7),new MeetingSchedule.Meeting("E",8,9)};
int lastEnd=int.MinValue;
foreach(var m in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
{
 bool accept=m.Start>=lastEnd;
 Console.WriteLine($"{m.Id} [{m.Start},{m.End}): {(accept?"accept":"reject")}");
 if(accept)lastEnd=m.End;
}
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


A feasible schedule can still have too few meetings. Choosing the earliest starting long meeting, for example, can block several shorter meetings. Always distinguish a feasibility invariant from an optimality argument.

## Exchange the First Choice

```{index} Exchange the First Choice
```

Let g be the globally earliest-finishing available meeting. Take an optimal schedule with first meeting o. Since end(g)≤end(o), replace o with g. Every later meeting starts at or after end(o), hence also at or after end(g); feasibility and count are unchanged. Thus some optimal schedule begins with g. Apply the same argument to the remaining compatible meetings.

The positive-duration assumption ensures g cannot also be a later meeting in the original schedule. After the exchange, the residual problem has the same shape. Induction on the number of available meetings establishes optimal count, rather than merely saying that g leaves room. The loop invariant can be strengthened: the accepted prefix can be extended to an optimal solution for the original instance.

## A Verified Counterexample for a Different Objective

```{index} A Verified Counterexample for a Different Objective
```

Change the goal from maximum count to maximum total meeting value. Earliest finish now fails: two short meetings can be worth less than one long meeting. Compute both feasible solutions rather than asserting a failure from an incomplete schedule.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",0,2,4),new MeetingSchedule.Meeting("B",2,4,4),new MeetingSchedule.Meeting("C",0,4,9)};
var chosen=MeetingSchedule.Select(meetings);
Console.WriteLine($"earliest finish: {string.Join(",",chosen.Select(m=>m.Id))}, count={chosen.Length}, value={chosen.Sum(m=>m.Value)}");
Console.WriteLine("alternative C: count=1, value=9; both schedules are feasible");
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


Count is maximized by A,B, but their value eight loses to C’s value nine. This does not refute the count algorithm; it refutes reusing it for weighted value. Chapter 24 develops dynamic programming for problems where a local commitment lacks this exchange property.

## Another Greedy Trap: Coin Counts

```{index} Another Greedy Trap: Coin Counts
```

For denominations 1,3,4 and amount six, choosing the largest affordable coin yields 4+1+1. The feasible alternative 3+3 uses fewer coins. Availability of coin one guarantees completion, not optimality.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int remaining=6;var coins=new List<int>();
foreach(int denomination in new[]{4,3,1})
 while(remaining>=denomination){coins.Add(denomination);remaining-=denomination;}
Console.WriteLine($"greedy={string.Join("+",coins)}, count={coins.Count}");
Console.WriteLine("alternative=3+3, count=2");

A greedy rule needs a proof tied to the denomination system and objective. A counterexample invalidates a universal claim; successful examples do not prove one. Test small complete alternatives, equal keys, empty inputs, and changed objectives before trusting a rule.

## Exercise

```{index} Exercise
```

Choose the maximum-count compatible schedule, including a negative start time and a back-to-back boundary. Explain the exchange argument and reject a zero-duration interval.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var meetings=new[]{new MeetingSchedule.Meeting("A",-3,0),new MeetingSchedule.Meeting("B",0,2),new MeetingSchedule.Meeting("C",-2,3)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
var meetings=new[]{new MeetingSchedule.Meeting("A",-3,0),new MeetingSchedule.Meeting("B",0,2),new MeetingSchedule.Meeting("C",-2,3)};
Console.WriteLine(string.Join(",",MeetingSchedule.Select(meetings).Select(m=>m.Id)));
try{MeetingSchedule.Select(new[]{new MeetingSchedule.Meeting("Bad",2,2)});}catch(ArgumentException){Console.WriteLine("zero duration rejected");}
public static class MeetingSchedule
{
    public record Meeting(string Id,int Start,int End,int Value=1);
    // Half-open, positive-duration intervals. Objective: maximum count.
    public static Meeting[] Select(Meeting[] meetings)
    {
        ArgumentNullException.ThrowIfNull(meetings);
        var ids=new HashSet<string>(StringComparer.Ordinal);
        foreach(var meeting in meetings)
            if(meeting is null||meeting.Id is null||!ids.Add(meeting.Id)||meeting.Start>=meeting.End)
                throw new ArgumentException("Unique IDs and Start < End required.");
        var chosen=new List<Meeting>();int lastEnd=int.MinValue;
        foreach(var meeting in meetings.OrderBy(m=>m.End).ThenBy(m=>m.Start).ThenBy(m=>m.Id,StringComparer.Ordinal))
            if(meeting.Start>=lastEnd){chosen.Add(meeting);lastEnd=meeting.End;}
        return chosen.ToArray();
    }
}


A,B
zero duration rejected


```{rubric} Footnotes
```
[^1]: Earliest-finish interval scheduling maximizes count under the fixed one-room, positive-duration model; weighted objectives require different reasoning.
[^2]: Greedy correctness requires more than feasibility. An exchange argument connects accepted choices to an optimal complete solution.